# Epzero quick start

`epzero.build()` configures an index for your data and your machine; this notebook shows what it chooses,
how to search and filter, and how to keep an index accurate as it changes.

```bash
pip install epzero
```

In [1]:
import numpy as np
import epzero

print("epzero", epzero.__version__, "on", epzero.machine())

epzero 0.4.0 on 13th Gen Intel(R) Core(TM) i5-13420H/avx2


## Data

Real embeddings put most of their variance in fewer directions than they have dimensions; this synthetic
set does too (384 dimensions, like a small sentence-embedding model), so it behaves like real data.

In [2]:
rng = np.random.default_rng(0)
basis = rng.normal(size=(48, 384))
centers = rng.normal(size=(200, 48))
def embed(n):
    z = centers[rng.integers(0, 200, n)] + 0.5 * rng.normal(size=(n, 48))
    return (z @ basis + 0.3 * rng.normal(size=(n, 384))).astype(np.float32)

vectors = embed(50_000)
queries = embed(200)  # unseen, from the same distribution
vectors.shape

(50000, 384)

## Build

No settings needed. `index.plan` lists every choice and why it was made.

In [3]:
%time index = epzero.build(vectors, metric="cosine")
print(index.plan)

CPU times: user 21.3 s, sys: 201 ms, total: 21.5 s
Wall time: 2.08 s
metric           'cosine'      
quantization     'sq8'           8-bit codes: a quarter of the memory traffic, re-ranked with the full vectors
M                16            
ef_construction  200           
nav_dim          32              30 principal directions hold 80% of the variance
fast_build       True            built on the codes: faster, and as good a graph in our measurements
refine           True            a second pass gave higher recall at equal speed on every dataset measured
fill_after       True            free link slots filled after the build (measured with principal directions)
half_vectors     False           off: 16-bit vectors halve their memory but cost speed and some recall; set it to save memory
rerank           300             up to 300: the walk on directions is approximate
rerank_z         3.0             stops the re-rank once the rest are 3 standard deviations past the k-th best
recall 

## Search

`search()` uses the beam width calibrated for the recall target (0.95 by default). `exact_search()` scans
every vector, the ground truth to measure recall against.

In [4]:
labels, distances = index.search(queries, k=10)
truth, _ = index.exact_search(queries, k=10, threads=0)
recall = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(labels, truth)])
print(f"recall@10 on unseen queries: {recall:.3f}")

recall@10 on unseen queries: 0.943


Any setting can be overridden, at build time or per search.

In [5]:
precise = epzero.build(vectors, metric="cosine", recall=0.99)
print("beam width for 0.95:", index.plan.ef, " for 0.99:", precise.plan.ef)
labels, _ = index.search(queries, k=50, ef=200)   # a per-call override
labels.shape

beam width for 0.95: 72  for 0.99: 120


(200, 50)

## Metadata filters

`epzero.Metadata` keeps fields per label and evaluates filters in the engine; the search then picks its
own strategy for the allowed set.

In [6]:
meta = epzero.Metadata()
langs = ["en", "de", "fr", "ja"]
meta.set_many(None, [{"lang": langs[i % 4], "year": 2015 + i % 10} for i in range(len(vectors))])
allowed = meta.select({"lang": ["en", "de"], "year": {"$gte": 2020}})
labels, _ = index.search(queries[:3], k=5, filter=index.filter(allowed))
print(len(allowed), "allowed;", [[meta.get(int(l)) for l in row[:2]] for row in labels])

12500 allowed; [[{'lang': 'en', 'year': 2021}, {'lang': 'de', 'year': 2020}], [{'lang': 'en', 'year': 2023}, {'lang': 'de', 'year': 2022}], [{'lang': 'de', 'year': 2024}, {'lang': 'en', 'year': 2023}]]


## After the build

An index keeps changing; `health()` says when it needs attention, and `recalibrate()` re-measures the
search on real queries without rebuilding. `add()` grows the index as needed.

In [7]:
index.add(embed(30_000), labels=np.arange(50_000, 80_000))
print(index.health()["advice"])
print(index.recalibrate(queries))
print(index.health()["advice"])

['recalibrate(queries): the index has grown 60% since its beam width was calibrated']


{'ef': 108, 'recall': 0.9509999752044678}
[]


## Save and load

The configuration is saved inside the index file; a loaded index searches with it again.

In [8]:
index.save("quickstart.epzero")
loaded = epzero.Index.load("quickstart.epzero")
print(loaded.plan.ef == index.plan.ef, len(loaded))

True 80000
